> **You do not need to run this notebook to reproduce the results.**
> It performs a fresh YouTube scrape and requires a personal YouTube Data API key (see **Prerequisites** below). If you only want to reproduce the analysis, **skip it**: the labelled dataset (`data/processed/labelled_comments_final.csv`) and the fixed train / validation / test splits are already included in the repository, and notebooks **05 onward** run directly from them. A fresh scrape will not recreate the exact historical corpus, because YouTube is a live platform; the frozen copies used in the reported analysis are included for exact verification.

# 01 - Collect YouTube comments (RQ1)

Collects comment threads (top-level comments and their replies) from ten 
manosphere-adjacent channels via the YouTube Data API v3, and writes them to
`data/raw/channel_comments.csv`.

**Prerequisites**
- Python 3.10+
- Packages: `google-api-python-client`, `httplib2`, `python-dotenv` (see `requirements.txt`)
- YouTube Data API key: copy `.env.example` to `.env` and set `YOUTUBE_API_KEY=...`

**Run from the `notebooks/` folder.** Collection is checkpointed after every
video, so if the daily quota is exhausted the notebook can be rerun to resume.

**Reproducibility note:** comments are requested in `relevance` order against the
API's live state, so a re-run returns a different set. The corpus used in the
study (351,813 comments, collected July 2026) is the frozen dataset; all
parameters below are fixed so the *procedure* is reproducible.

In [ ]:
# Imports and API setup
import csv
import json
import time
import random
import logging
import socket
import ssl
import http.client as http_client
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import httplib2
from dotenv import dotenv_values
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

API_KEY = dotenv_values(PROJECT_ROOT / ".env").get("YOUTUBE_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "YouTube API key not found. Copy .env.example to .env and add "
        "YOUTUBE_API_KEY=your_key_here"
    )

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("harvest")

youtube = build(
    "youtube", "v3",
    developerKey=API_KEY,
    http=httplib2.Http(timeout=120),
    cache_discovery=False,
)
print("YouTube API client ready")

In [ ]:
# RQ1 collection configuration

FIELDNAMES = [
    "comment_id", "channel_handle", "channel_id", "video_id", "parent_id",
    "is_reply", "author", "text", "like_count", "published_at",
]


@dataclass(frozen=True)
class Config:
    max_comments_per_video: int = 200
    max_videos_per_channel: int = 500
    target_comments_per_channel: int = 40_000
    include_replies: bool = True          # RQ1 retains top-level comments and their replies
    comment_order: str = "relevance"
    max_retries: int = 8

    output_file: Path = PROJECT_ROOT / "data" / "raw" / "channel_comments.csv"
    checkpoint_file: Path = PROJECT_ROOT / "data" / "raw" / ".harvest_checkpoint.json"
    resolve_cache: Path = PROJECT_ROOT / "data" / "raw" / "resolved_channels.json"


class QuotaExceeded(Exception):
    """Raised when the YouTube API daily quota is exhausted."""

In [ ]:
# API error handling

def _reason(error: HttpError) -> str:
    """Extract the YouTube API error reason where available."""
    try:
        payload = json.loads(error.content.decode("utf-8"))
        return payload["error"]["errors"][0].get("reason", "")
    except (AttributeError, KeyError, IndexError, TypeError, ValueError):
        return ""


def execute(request, cfg: Config) -> Optional[dict]:
    """Execute an API request with retry and quota handling."""

    unavailable_reasons = {
        "commentsDisabled",
        "videoNotFound",
        "playlistNotFound",
        "channelNotFound",
    }

    quota_reasons = {
        "quotaExceeded",
        "dailyLimitExceeded",
    }

    transient_reasons = {
        "rateLimitExceeded",
        "userRateLimitExceeded",
        "backendError",
        "processingFailure",
    }

    transient_statuses = {429, 500, 502, 503, 504}

    for attempt in range(cfg.max_retries):
        try:
            return request.execute()

        except HttpError as error:
            reason = _reason(error)
            status = error.resp.status

            if reason in quota_reasons:
                raise QuotaExceeded(
                    "YouTube API quota exhausted."
                ) from error

            if reason in unavailable_reasons:
                log.warning(
                    "Skipping unavailable resource (%s).",
                    reason,
                )
                return None

            is_transient = (
                status in transient_statuses
                or reason in transient_reasons
            )

            if is_transient:
                if attempt < cfg.max_retries - 1:
                    wait = (2 ** attempt) + random.random()

                    log.warning(
                        "API error (%s); retrying in %.1f seconds.",
                        reason or status,
                        wait,
                    )

                    time.sleep(wait)
                    continue

                # Some videos repeatedly return processingFailure.
                # Skip that individual resource after all attempts.
                if reason == "processingFailure":
                    log.warning(
                        "Skipping resource after repeated "
                        "processing failures."
                    )
                    return None

            # Do not conceal unexpected authentication,
            # permission or request errors.
            raise

        except (
            TimeoutError,
            socket.timeout,
            ssl.SSLError,
            ConnectionResetError,
            http_client.HTTPException,
            OSError,
        ) as error:
            if attempt < cfg.max_retries - 1:
                wait = (2 ** attempt) + random.random()

                log.warning(
                    "Network error (%s); retrying in %.1f seconds.",
                    type(error).__name__,
                    wait,
                )

                time.sleep(wait)
                continue

            raise

    return None

In [ ]:
# Resolve channel information from the YouTube API

def _channel_item(
    youtube,
    cfg: Config,
    **lookup,
) -> Optional[dict]:
    """Retrieve one channel using an ID or handle."""

    response = execute(
        youtube.channels().list(
            part="snippet,statistics,contentDetails",
            **lookup,
        ),
        cfg,
    )

    items = (response or {}).get("items", [])
    return items[0] if items else None


def resolve_channel(
    youtube,
    entry: dict,
    cfg: Config,
) -> Optional[dict]:
    """Resolve a configured channel to its stable YouTube metadata."""

    if entry.get("channel_id"):
        item = _channel_item(
            youtube,
            cfg,
            id=entry["channel_id"],
        )

    elif entry.get("handle"):
        item = _channel_item(
            youtube,
            cfg,
            forHandle=entry["handle"].lstrip("@"),
        )

    else:
        raise ValueError(
            f"{entry.get('name', 'Channel')} requires either "
            "'channel_id' or 'handle'."
        )

    if item is None:
        return None

    statistics = item.get("statistics", {})

    return {
        "name": entry["name"],
        "channel_id": item["id"],
        "uploads_playlist": (
            item["contentDetails"]
            ["relatedPlaylists"]
            ["uploads"]
        ),
        "title": item["snippet"]["title"],
        "subscribers": int(
            statistics.get("subscriberCount", 0)
        ),
        "videos": int(
            statistics.get("videoCount", 0)
        ),
    }


def resolve_all(
    youtube,
    candidates: list[dict],
    cfg: Config,
) -> list[dict]:
    """Resolve all configured channels and save their metadata."""

    resolved = []
    unresolved = []

    for entry in candidates:
        channel = resolve_channel(
            youtube,
            entry,
            cfg,
        )

        if channel is None:
            unresolved.append(entry["name"])
            continue

        log.info(
            "%-22s -> %-26s %9d subscribers / %d videos",
            channel["name"],
            channel["title"][:26],
            channel["subscribers"],
            channel["videos"],
        )

        resolved.append(channel)

    if unresolved:
        raise RuntimeError(
            "The following channels could not be resolved: "
            + ", ".join(unresolved)
        )

    cfg.resolve_cache.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    cfg.resolve_cache.write_text(
        json.dumps(resolved, indent=2),
        encoding="utf-8",
    )

    return resolved

In [ ]:
# Retrieve video IDs and comments from the YouTube API

def get_video_ids(youtube, uploads_playlist: str, limit: int, cfg: Config) -> list[str]:
    """Return up to `limit` video IDs from a channel's uploads playlist."""
    ids: list[str] = []
    token = None
    while len(ids) < limit:
        response = execute(
            youtube.playlistItems().list(
                part="contentDetails",
                playlistId=uploads_playlist,
                maxResults=min(50, limit - len(ids)),
                pageToken=token,
            ),
            cfg,
        )
        if not response:
            break
        ids.extend(item["contentDetails"]["videoId"] for item in response["items"])
        token = response.get("nextPageToken")
        if not token:
            break
    return ids[:limit]


def _row(comment_id, channel, video_id, parent, is_reply, snippet) -> dict:
    """Build one output row from a comment snippet."""
    return {
        "comment_id": comment_id,
        "channel_handle": channel["name"],
        "channel_id": channel["channel_id"],
        "video_id": video_id,
        "parent_id": parent or "",
        "is_reply": is_reply,
        "author": snippet.get("authorDisplayName", ""),
        "text": snippet.get("textOriginal", snippet.get("textDisplay", "")),
        "like_count": snippet.get("likeCount", 0),
        "published_at": snippet.get("publishedAt", ""),
    }


def get_comments(youtube, video_id: str, channel: dict, limit: int, cfg: Config) -> list[dict]:
    """Return up to `limit` comments for a video (top-level plus replies when enabled)."""
    rows: list[dict] = []
    token = None
    while len(rows) < limit:
        response = execute(
            youtube.commentThreads().list(
                part="snippet,replies" if cfg.include_replies else "snippet",
                videoId=video_id,
                maxResults=min(100, limit - len(rows)),
                order=cfg.comment_order,
                pageToken=token,
                textFormat="plainText",
            ),
            cfg,
        )
        if response is None:
            break
        for thread in response.get("items", []):
            top = thread["snippet"]["topLevelComment"]
            rows.append(_row(top["id"], channel, video_id, None, False, top["snippet"]))
            if cfg.include_replies and "replies" in thread:
                for reply in thread["replies"]["comments"]:
                    rows.append(_row(reply["id"], channel, video_id, top["id"], True, reply["snippet"]))
            if len(rows) >= limit:
                break
        token = response.get("nextPageToken")
        if not token:
            break
    return rows[:limit]

In [ ]:
# Harvest comments and save them incrementally

def _load_done(cfg: Config) -> set[str]:
    """
    Load the identifiers of videos already processed.

    Each checkpoint entry combines the channel ID and video ID.
    This prevents completed videos from being requested again when
    collection resumes after interruption or quota exhaustion.
    """

    if not cfg.checkpoint_file.exists():
        return set()

    completed = json.loads(
        cfg.checkpoint_file.read_text(
            encoding="utf-8"
        )
    )

    return set(completed)


def _load_existing_channel_counts(
    cfg: Config,
) -> dict[str, int]:
    """
    Count comments already saved for each channel.

    These counts ensure that a resumed collection continues towards,
    but does not exceed, the configured per-channel target.
    """

    counts = {}

    if not cfg.output_file.exists():
        return counts

    with cfg.output_file.open(
        "r",
        newline="",
        encoding="utf-8",
    ) as file:
        reader = csv.DictReader(file)

        for row in reader:
            channel_name = row.get("channel_handle")

            if channel_name:
                counts[channel_name] = (
                    counts.get(channel_name, 0) + 1
                )

    return counts


def harvest(
    youtube,
    channels: list[dict],
    cfg: Config,
) -> int:
    """
    Collect comments for each resolved channel.

    Results are appended to a CSV file and a checkpoint is saved
    after every completed video. The function can therefore be
    rerun to resume collection without repeating completed videos.
    """

    # Create the output directories if they do not already exist.
    cfg.output_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    cfg.checkpoint_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Restore progress from any previous collection run.
    completed_videos = _load_done(cfg)
    channel_counts = _load_existing_channel_counts(cfg)

    # A header is written only when the output file is first created.
    new_file = not cfg.output_file.exists()

    # This counts only comments added during the current run.
    total_written = 0

    with cfg.output_file.open(
        "a",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=FIELDNAMES,
        )

        if new_file:
            writer.writeheader()

        try:
            for channel in channels:
                channel_name = channel["name"]

                # Start from the number of comments already saved
                # for this channel during earlier runs.
                channel_written = channel_counts.get(
                    channel_name,
                    0,
                )

                # Skip channels that have already reached the
                # configured 40,000-comment target.
                if (
                    channel_written
                    >= cfg.target_comments_per_channel
                ):
                    log.info(
                        "%s: already has %d comments; skipping.",
                        channel_name,
                        channel_written,
                    )
                    continue

                # Retrieve up to the configured number of videos
                # from the channel's uploads playlist.
                video_ids = get_video_ids(
                    youtube=youtube,
                    uploads_playlist=(
                        channel["uploads_playlist"]
                    ),
                    limit=cfg.max_videos_per_channel,
                    cfg=cfg,
                )

                log.info(
                    "%s: %d videos selected.",
                    channel_name,
                    len(video_ids),
                )

                for video_id in video_ids:
                    # Stop requesting videos once the channel-level
                    # comment target has been reached
                    if (
                        channel_written
                        >= cfg.target_comments_per_channel
                    ):
                        log.info(
                            "%s: reached target of %d comments.",
                            channel_name,
                            cfg.target_comments_per_channel,
                        )
                        break

                    # Combining channel ID and video ID creates a
                    # stable checkpoint identifier.
                    checkpoint_key = (
                        f"{channel['channel_id']}:{video_id}"
                    )

                    # Skip videos completed during an earlier run.
                    if checkpoint_key in completed_videos:
                        continue

                    # Where fewer than 200 comments remain before the
                    # channel target, request only the remaining number.
                    remaining = (
                        cfg.target_comments_per_channel
                        - channel_written
                    )

                    per_video_limit = min(
                        cfg.max_comments_per_video,
                        remaining,
                    )

                    # Retrieve top-level comments for this video
                    rows = get_comments(
                        youtube=youtube,
                        video_id=video_id,
                        channel=channel,
                        limit=per_video_limit,
                        cfg=cfg,
                    )

                    # Save immediately so completed work is retained
                    # if collection is interrupted later
                    writer.writerows(rows)
                    file.flush()

                    total_written += len(rows)
                    channel_written += len(rows)

                    channel_counts[channel_name] = (
                        channel_written
                    )

                    # Mark the video as complete only after its rows
                    # have been written successfully
                    completed_videos.add(checkpoint_key)

                    cfg.checkpoint_file.write_text(
                        json.dumps(
                            sorted(completed_videos),
                            indent=2,
                        ),
                        encoding="utf-8",
                    )

                    log.info(
                        "%s | %s | %d comments | "
                        "channel total: %d",
                        channel_name,
                        video_id,
                        len(rows),
                        channel_written,
                    )

        except QuotaExceeded:
            # Previously written comments and the latest checkpoint
            # remain available, allowing collection to resume later.
            log.warning(
                "Quota exhausted—%d new comments saved. "
                "Rerun later to resume.",
                total_written,
            )

    log.info(
        "Collection complete: %d new comments written to %s",
        total_written,
        cfg.output_file,
    )

    return total_written

In [ ]:
# Define the RQ1 channels and run collection

RQ1_CHANNELS = [
    {
        "name": "The Free Agent Lifestyle",
        "handle": "@TheFreeAgentLifestyle",
    },
    {
        "name": "Myron Gaines X",
        "handle": "@MyronGainesX",
    },
    {
        "name": "Fresh and Fit",
        "handle": "@FreshFitMiami",
    },
    {
        "name": "Fresh and Fit Clips",
        "handle": "@FreshandFitClips",
    },
    {
        "name": "Rational Male Clips",
        "handle": "@RationalMaleClips",
    },
    {
        "name": "Rich Cooper Clips",
        "handle": "@RichCooperClips",
    },
    {
        "name": "Better Bachelor",
        "channel_id": "UC9ctsJZ2aD1nCexfqj342NQ",
    },
    {
        "name": "21 Studios",
        "channel_id": "UCuErSr7xeR763BzTJL7yJ7A",
    },
    {
        "name": "Rollo Tomassi",
        "channel_id": "UC8eLDzfH6YfZRDbDuMsNWiA",
    },
    {
        "name": "Entrepreneurs in Cars",
        "channel_id": "UCuKqtwPO5ci5_KoZGbfl33g",
    },
]


# Use the collection settings defined earlier.
cfg = Config()


# Resolve the supplied channel IDs and handles to their
# current YouTube channel metadata and uploads playlists.
resolved_channels = resolve_all(
    youtube=youtube,
    candidates=RQ1_CHANNELS,
    cfg=cfg,
)


# Collect comments and save them to data/raw/channel_comments.csv.
# If collection stops because of quota exhaustion, rerunning this
# cell resumes from the saved checkpoint.
new_comments = harvest(
    youtube=youtube,
    channels=resolved_channels,
    cfg=cfg,
)


print(
    f"Collection run complete: "
    f"{new_comments:,} new comments saved."
)